# 🤖 Inference Comparison: Base vs DAPT Fine-Tuned Model
This notebook compares the outputs of the base DeepSeek model and your domain-adapted version using real university-related questions.

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import pandas as pd
import torch
from peft import PeftModel
import gc
import os

# === CONFIG ===
BASE_MODEL_8B = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
DAPT_ADAPTER_PATH_8B = "dapt_qlora_model_output"
DAPT_MODEL_PATH_1_3B = "dapt_deepseek_1.3b_model_output"
QUESTIONS_FILE = "../healthandsafety_eval_questions.txt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LOAD_IN_4BIT_INFERENCE_8B = False
compute_dtype_4bit = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

dtype_1_3b = torch.float16

In [ ]:
# --- Function to clear memory ---
def clear_memory():
    """Clears GPU cache and runs Python garbage collector."""
    print("🧹 Clearing CUDA cache and collecting garbage...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("✅ Memory cleared.")

# Dictionary to hold loaded models and pipelines
models_pipelines = {}

if not os.path.exists(DAPT_ADAPTER_PATH_8B):
    print(f"⚠️ WARNING: DAPT 8B adapter path not found: {DAPT_ADAPTER_PATH_8B}")
if not os.path.exists(DAPT_MODEL_PATH_1_3B):
    print(f"⚠️ WARNING: DAPT 1.3B model path not found: {DAPT_MODEL_PATH_1_3B}")

# --- Load Base 8B Model ---
print(f"--- Loading Base Model ({BASE_MODEL_8B}) ---")
bnb_config_inference_8b = None
if LOAD_IN_4BIT_INFERENCE_8B:
    print(f"⚙️ Configuring 4-bit quantization for 8B inference (compute dtype: {compute_dtype_4bit})...")
    bnb_config_inference_8b = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype_4bit,
    )

try:
    base_tokenizer_8b = AutoTokenizer.from_pretrained(BASE_MODEL_8B, trust_remote_code=True)
    if base_tokenizer_8b.pad_token is None: base_tokenizer_8b.pad_token = base_tokenizer_8b.eos_token

    base_model_8b = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_8B,
        quantization_config=bnb_config_inference_8b,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype="auto" 
    )
    print("Creating pipeline for base 8B model...")
    models_pipelines['base_8b'] = {
        "pipeline": pipeline("text-generation", model=base_model_8b, tokenizer=base_tokenizer_8b),
        "model_ref": base_model_8b
    }
    print("✅ Base 8B model loaded.")
except Exception as e:
    print(f"❌ Error loading Base 8B model: {e}")
clear_memory()

if os.path.exists(DAPT_ADAPTER_PATH_8B):
    print(f"--- Loading DAPT 8B Model (Adapters: {DAPT_ADAPTER_PATH_8B}) ---")
    try:
        dapt_base_model_8b = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_8B,
            quantization_config=bnb_config_inference_8b,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype="auto"
        )
        print(f"Applying LoRA adapters from {DAPT_ADAPTER_PATH_8B}...")
        dapt_model_with_adapters_8b = PeftModel.from_pretrained(dapt_base_model_8b, DAPT_ADAPTER_PATH_8B)
        dapt_tokenizer_8b = AutoTokenizer.from_pretrained(BASE_MODEL_8B, trust_remote_code=True)
        if dapt_tokenizer_8b.pad_token is None: dapt_tokenizer_8b.pad_token = dapt_tokenizer_8b.eos_token

        print("Creating pipeline for DAPT 8B model...")
        models_pipelines['dapt_8b_qlora'] = {
             "pipeline": pipeline("text-generation", model=dapt_model_with_adapters_8b, tokenizer=dapt_tokenizer_8b),
             "model_ref": dapt_model_with_adapters_8b
        }
        print("✅ DAPT 8B model loaded.")
    except Exception as e:
        print(f"❌ Error loading DAPT 8B model: {e}")
    clear_memory() # Clear memory after loading

# --- Load DAPT 1.3B Model (Full Model) ---
if os.path.exists(DAPT_MODEL_PATH_1_3B):
    print(f"--- Loading DAPT 1.3B Model ({DAPT_MODEL_PATH_1_3B}) ---")
    try:
        dapt_tokenizer_1_3b = AutoTokenizer.from_pretrained(DAPT_MODEL_PATH_1_3B, trust_remote_code=True)
        dapt_model_1_3b = AutoModelForCausalLM.from_pretrained(
            DAPT_MODEL_PATH_1_3B,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=dtype_1_3b # Load in fp16
        )
        print("Creating pipeline for DAPT 1.3B model...")
        models_pipelines['dapt_1.3b_fp16'] = {
            "pipeline": pipeline("text-generation", model=dapt_model_1_3b, tokenizer=dapt_tokenizer_1_3b),
            "model_ref": dapt_model_1_3b 
        }
        print("✅ DAPT 1.3B model loaded.")
    except Exception as e:
        print(f"❌ Error loading DAPT 1.3B model: {e}")
    clear_memory() # Clear memory after loading

print(f"--- Models ready for inference: {list(models_pipelines.keys())} ---")



In [ ]:
# Load evaluation questions
try:
    with open(QUESTIONS_FILE, "r", encoding="utf-8") as f:
        questions = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(questions)} questions from {QUESTIONS_FILE}")
    print("First 3 questions:", questions[:3])
except FileNotFoundError:
    print(f"❌ Error: Questions file not found at {QUESTIONS_FILE}")
    questions = [] 

In [ ]:
# Run all loaded models on the questions
instruction = (
    "You are a member of the University of Bradford staff. "
    "Answer student questions based only on the information you’ve been trained on. "
    "If you are unsure, say you don’t know. Do not make up answers.\n\n"
    "Question: {question}\n" 
    "Answer:" 
)

default_pad_token_id = base_tokenizer_8b.eos_token_id if 'base_8b' in models_pipelines else 50256 

generation_params = {
    "max_new_tokens": 150,
    "do_sample": False, 
    "pad_token_id": default_pad_token_id,
    "eos_token_id": default_pad_token_id
}

results = []
print("🚀 Starting inference comparison...")

# Function to safely generate and extract answer
def get_model_answer(model_key, prompt, gen_params):
    """Generates text using a pipeline and extracts the answer part."""
    if model_key not in models_pipelines:
        print(f"Model {model_key} not loaded, skipping.")
        return "Model not loaded"
    try:
        pipe_info = models_pipelines[model_key]
        current_params = gen_params.copy()
        current_params["pad_token_id"] = pipe_info["pipeline"].tokenizer.eos_token_id \
                                         if pipe_info["pipeline"].tokenizer.eos_token_id is not None \
                                         else default_pad_token_id
        current_params["eos_token_id"] = current_params["pad_token_id"]

        print(f"Generating with {model_key}...")
        output_full = pipe_info["pipeline"](prompt, **current_params)[0]["generated_text"]
        # Extract text after "Answer:"
        answer_part = output_full.split("Answer:")
        if len(answer_part) > 1:
            answer = answer_part[1].strip()
        else:

            prompt_base = prompt.split("Answer:")[0]
            answer = output_full.replace(prompt_base, "").strip()

        return answer
    except Exception as e:
        print(f"❌ Error generating with {model_key}: {e}")
        return f"ERROR: {e}"
    finally:
        clear_memory() # Clear memory after each model generation

# --- Inference Loop ---
for i, q in enumerate(questions):
    print(f"\n--- Processing question {i+1}/{len(questions)}: '{q}' ---")
    prompt = instruction.format(question=q)
    current_result = {"Question": q}

    # Generate with each loaded model
    current_result["Base Model (8B)"] = get_model_answer('base_8b', prompt, generation_params)
    current_result["DAPT Model (8B QLoRA)"] = get_model_answer('dapt_8b_qlora', prompt, generation_params)
    current_result["DAPT Model (1.3B FP16)"] = get_model_answer('dapt_1.3b_fp16', prompt, generation_params)

    results.append(current_result)
    print("-" * 30) # Separator

print("✅ Inference complete.")

# --- Display Results ---
df = pd.DataFrame(results)

# Define desired column order
column_order = ["Question", "Base Model (8B)", "DAPT Model (8B QLoRA)", "DAPT Model (1.3B FP16)"]
# Filter out columns that might not exist if a model failed to load
existing_columns = [col for col in column_order if col in df.columns]
df = df[existing_columns]


print(df.to_markdown(index=False))

In [ ]:
try:
    output_csv_file = "model_comparison_output_3models.csv"
    df.to_csv(output_csv_file, index=False)
    print(f"✅ Results saved to {output_csv_file}")
except Exception as e:
    print(f"❌ Error saving results to CSV: {e}")